# Smart MCQ Solver Challenge (DL & GenAI Project)

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


## Load the datasets

In [48]:
df1 = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
df2 = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

In [49]:
print(df1.shape)
print(df2.shape)
print(sample.shape)

(2000, 8)
(500, 7)
(500, 2)


## Import libraries

In [50]:

# Smart MCQ Solver
# DistilBERT Transformer Baseline


# Standard Library
import os
import random
import warnings

warnings.filterwarnings("ignore")


# Data Manipulation
import numpy as np
import pandas as pd


# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)


# PyTorch
import torch


# Hugging Face
import transformers
import datasets

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)


# Utilities
from scipy.special import softmax
from tqdm.auto import tqdm


# Display Versions
print("Environment Information")

print(f"Transformers : {transformers.__version__}")
print(f"Datasets     : {datasets.__version__}")
print(f"PyTorch      : {torch.__version__}")

print("\nGPU Available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))

Environment Information
Transformers : 5.0.0
Datasets     : 4.8.5
PyTorch      : 2.10.0+cu128

GPU Available : True
GPU : Tesla T4


## Set Configurations

In [51]:
# Configuration

class CFG:

    # Model
    MODEL_NAME = "distilbert-base-uncased"

    # Future notebooks:
    # MODEL_NAME = "FacebookAI/roberta-base"
    # MODEL_NAME = "microsoft/deberta-v3-small"


    # Labels
    NUM_LABELS = 5

    LABEL2ID = {
        "A": 0,
        "B": 1,
        "C": 2,
        "D": 3,
        "E": 4,
    }

    ID2LABEL = {v: k for k, v in LABEL2ID.items()}


    # Tokenization
    MAX_LENGTH = 128

    
    # Training
    TRAIN_BATCH_SIZE = 16

    VALID_BATCH_SIZE = 32

    LEARNING_RATE = 2e-5

    WEIGHT_DECAY = 0.01

    EPOCHS = 5

    WARMUP_RATIO = 0.10

    
    # Misc
    VALID_SIZE = 0.20

    RANDOM_STATE = 42

    NUM_WORKERS = 2

In [52]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Device: cuda
Tesla T4


In [53]:
train = df1.copy()
test = df2.copy()

In [54]:
### Reproducibility

def seed_everything(seed=42):

    random.seed(seed)

    np.random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True

    torch.backends.cudnn.benchmark = False


seed_everything(CFG.RANDOM_STATE)

In [55]:
# Missing Values

print("Train Missing Values")

display(train.isnull().sum())

print()

print("Test Missing Values")

display(test.isnull().sum())

Train Missing Values


id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64


Test Missing Values


id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
dtype: int64

In [56]:
# Label Distribution

print(train["answer"].value_counts())

print()

print(
    train["answer"]
    .value_counts(normalize=True)
    .sort_index()
)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

answer
A    0.1845
B    0.2450
C    0.2295
D    0.1790
E    0.1620
Name: proportion, dtype: float64


In [57]:
## Dataset Summary

print("\nTrain Columns")

print(train.columns.tolist())

print("\nTest Columns")

print(test.columns.tolist())

print("\nSubmission Columns")

print(sample.columns.tolist())


Train Columns
['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer']

Test Columns
['id', 'prompt', 'A', 'B', 'C', 'D', 'E']

Submission Columns
['ID', 'Prediction']


## Dataset preprocessing

In [58]:
# Label Encoding

train["label"] = train["answer"].map(CFG.LABEL2ID)

print(train[["answer", "label"]].head())

  answer  label
0      B      1
1      A      0
2      C      2
3      B      1
4      A      0


## Build Input Text

This is one of the most important steps.

Instead of aggressive preprocessing, preserve the natural language and combine the prompt with all answer choices into a single text sequence.

In [59]:
# Build Input Text
def build_input_text(row):
    return (
        f"Question: {row['prompt']} "
        f"[SEP] A: {row['A']} "
        f"[SEP] B: {row['B']} "
        f"[SEP] C: {row['C']} "
        f"[SEP] D: {row['D']} "
        f"[SEP] E: {row['E']}"
    )

In [60]:
train["text"] = train.apply(build_input_text, axis=1)
test["text"] = test.apply(build_input_text, axis=1)

In [61]:
print(train["text"].iloc[0])

Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. [SEP] B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement. [SEP] C: Martin Heidegger does not believe in the existence of time or that it has any effect on human consciousness. The relationship to the past and the future is insignificant, and human existence is solely based on the present. [SEP] D: Martin Heidegger beli

## Train & Validation Split

In [62]:
# Train / Validation Split

train_df, valid_df = train_test_split(
    train,
    test_size=CFG.VALID_SIZE,
    random_state=CFG.RANDOM_STATE,
    stratify=train["label"],
)

print("Train :", train_df.shape)
print("Valid :", valid_df.shape)

Train : (1600, 10)
Valid : (400, 10)


## Tokenization

In [63]:
### Load the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    CFG.MODEL_NAME
)

print(tokenizer)

BertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


In [64]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [65]:
### Tokenization Function

### We tokenize the combined question + options text.


def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=CFG.MAX_LENGTH,
    )

In [66]:
#### Test the Tokenizer
sample_tokens = tokenizer(
    train_df.iloc[0]["text"]
)

print(sample_tokens.keys())

KeysView({'input_ids': [101, 3160, 1024, 7276, 1996, 2087, 8321, 5724, 1024, 2054, 2003, 6310, 8446, 2937, 10949, 1006, 12256, 2094, 1007, 1029, 2013, 1996, 2206, 9804, 1012, 102, 1037, 1024, 12256, 2094, 2003, 1037, 6958, 2008, 7607, 1996, 5248, 1997, 2422, 1999, 1996, 3739, 1997, 2844, 19790, 4249, 1012, 2009, 2003, 2019, 4522, 2000, 1996, 10744, 1997, 2601, 3043, 1999, 3408, 1997, 9990, 2339, 21706, 2079, 2025, 3711, 2000, 15470, 1996, 2747, 5319, 4277, 1997, 5584, 1012, 102, 1038, 1024, 12256, 2094, 2003, 1037, 10744, 2008, 17146, 1037, 14080, 1997, 15313, 1005, 1055, 7705, 1997, 2236, 20805, 2000, 4070, 2005, 5159, 5144, 1997, 21706, 1012, 2009, 2003, 2019, 4522, 2000, 1996, 10744, 1997, 2601, 3043, 1999, 3408, 1997, 9990, 2339, 21706, 2079, 2025, 3711, 2000, 15470, 1996, 2747, 5319, 4277, 1997, 5584, 1012, 102, 1039, 1024, 12256, 2094, 2003, 1037, 10744, 2008, 17146, 1037, 14080, 1997, 8446, 1005, 1055, 2375, 1997, 5415, 24665, 18891, 12516, 2000, 4070, 2005, 5159, 5144, 1997, 21

## Hugging Face dataset

In [67]:
#### Convert Pandas DataFrame to Hugging Face Dataset

train_dataset = Dataset.from_pandas(
    train_df[["text", "label"]],
    preserve_index=False,
)

valid_dataset = Dataset.from_pandas(
    valid_df[["text", "label"]],
    preserve_index=False,
)

test_dataset = Dataset.from_pandas(
    test[["text"]],
    preserve_index=False,
)

In [68]:
### Tokenize Datasets
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True,
)

valid_dataset = valid_dataset.map(
    tokenize_function,
    batched=True,
)

test_dataset = test_dataset.map(
    tokenize_function,
    batched=True,
)

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [69]:
## Remove Unnecessary Columns
### Remove Raw Text
train_dataset = train_dataset.remove_columns(
    ["text"]
)

valid_dataset = valid_dataset.remove_columns(
    ["text"]
)

test_dataset = test_dataset.remove_columns(
    ["text"]
)

In [70]:
### Rename Label Column

## Hugging Face expects the target column to be named labels.

train_dataset = train_dataset.rename_column(
    "label",
    "labels"
)

valid_dataset = valid_dataset.rename_column(
    "label",
    "labels"
)

## Set Dataset Format

In [71]:
## Set Dataset Format
train_dataset.set_format("torch")
valid_dataset.set_format("torch")
test_dataset.set_format("torch")

In [73]:
## verify dataset
print(train_dataset)

print(valid_dataset)

print(test_dataset)

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1600
})
Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 400
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 500
})


## Load DistilBERT Model

In [74]:
## Load DistilBERT Model
model = AutoModelForSequenceClassification.from_pretrained(
    CFG.MODEL_NAME,
    num_labels=CFG.NUM_LABELS,
    id2label=CFG.ID2LABEL,
    label2id=CFG.LABEL2ID,
)

model.to(device)

print(model.__class__.__name__)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification


In [75]:
### Verify the Model
print(model.config)

DistilBertConfig {
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": null,
  "dim": 768,
  "dropout": 0.1,
  "dtype": "float32",
  "eos_token_id": null,
  "hidden_dim": 3072,
  "id2label": {
    "0": "A",
    "1": "B",
    "2": "C",
    "3": "D",
    "4": "E"
  },
  "initializer_range": 0.02,
  "label2id": {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
  },
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "tie_word_embeddings": true,
  "transformers_version": "5.0.0",
  "vocab_size": 30522
}



## Evaluation Metric (MAP@3)

In [76]:
def mapk(actual, predicted, k=3):
    """
    actual    : integer label
    predicted : ranked label indices
    """

    score = 0.0

    for i, p in enumerate(predicted[:k]):
        if p == actual:
            score = 1.0 / (i + 1)
            break

    return score

In [77]:
## That will print both accuracy and MAP@3 score

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    probs = softmax(logits, axis=1)

    ranked = np.argsort(-probs, axis=1)

    scores = [
        mapk(y, pred, k=3)
        for y, pred in zip(labels, ranked)
    ]

    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "map@3": np.mean(scores),
    }

## Training DistilBERT

Now we'll build the training pipeline.

In [81]:
## Training Arguments

## Since you're on Kaggle and short on time, I recommend these settings as a solid starting point.

training_args = TrainingArguments(

    output_dir="./results",

    # Required
    do_train=True,
    do_eval=True,

    # Evaluation
    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="map@3",
    greater_is_better=True,

    # Hyperparameters
    learning_rate=CFG.LEARNING_RATE,
    per_device_train_batch_size=CFG.TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=CFG.VALID_BATCH_SIZE,

    num_train_epochs=CFG.EPOCHS,

    weight_decay=CFG.WEIGHT_DECAY,
    warmup_ratio=CFG.WARMUP_RATIO,

    # Logging
    logging_steps=20,

    # Saving
    save_total_limit=2,

    # Performance
    fp16=torch.cuda.is_available(),

    # Reproducibility
    seed=CFG.RANDOM_STATE,

    # Disable online logging
    report_to="none",
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [82]:
from transformers import TrainingArguments
import inspect

print(inspect.signature(TrainingArguments.__init__))

(self, output_dir: str | None = None, do_train: bool = False, do_eval: bool = False, do_predict: bool = False, eval_strategy: transformers.trainer_utils.IntervalStrategy | str = 'no', prediction_loss_only: bool = False, per_device_train_batch_size: int = 8, per_device_eval_batch_size: int = 8, gradient_accumulation_steps: int = 1, eval_accumulation_steps: int | None = None, eval_delay: float = 0, torch_empty_cache_steps: int | None = None, learning_rate: float = 5e-05, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, max_grad_norm: float = 1.0, num_train_epochs: float = 3.0, max_steps: int = -1, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_ratio: float | None = None, warmup_steps: float = 0, log_level: str = 'passive', log_level_replica: str = 'warning', log_on_each_node: bool = True, logging_dir: str | None = None, logging_strategy: transfor

In [83]:
### Trainer
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=valid_dataset,

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics,
)

In [84]:
## Train model
train_result = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Map@3
1,3.130502,2.842700,0.455000,0.617500
2,1.980912,1.628939,0.840000,0.907917
3,1.094976,0.727882,0.962500,0.975833
4,0.437495,0.308258,1.000000,1.000000
5,0.300861,0.221406,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


## Evaluation

In [90]:
results = trainer.evaluate()

for key, value in results.items():
    print(f"{key:25s}: {value}")

eval_loss                : 0.30825138092041016
eval_accuracy            : 1.0
eval_map@3               : 1.0
eval_runtime             : 0.9719
eval_samples_per_second  : 411.575
eval_steps_per_second    : 7.203
epoch                    : 5.0


In [87]:
## Save the best model
trainer.save_model("./best_distilbert")

tokenizer.save_pretrained("./best_distilbert")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./best_distilbert/tokenizer_config.json', './best_distilbert/tokenizer.json')

In [91]:
valid_predictions = trainer.predict(valid_dataset)

valid_logits = valid_predictions.predictions

valid_probs = softmax(valid_logits, axis=1)

valid_pred_labels = np.argmax(valid_probs, axis=1)

valid_true_labels = valid_df["label"].values

In [92]:
print("Validation Accuracy")
print("=" * 40)

accuracy = accuracy_score(
    valid_true_labels,
    valid_pred_labels
)

print(f"Accuracy : {accuracy:.5f}")

Validation Accuracy
Accuracy : 1.00000


In [93]:
## Classification reports
print(
    classification_report(
        valid_true_labels,
        valid_pred_labels,
        target_names=["A", "B", "C", "D", "E"]
    )
)

              precision    recall  f1-score   support

           A       1.00      1.00      1.00        74
           B       1.00      1.00      1.00        98
           C       1.00      1.00      1.00        92
           D       1.00      1.00      1.00        71
           E       1.00      1.00      1.00        65

    accuracy                           1.00       400
   macro avg       1.00      1.00      1.00       400
weighted avg       1.00      1.00      1.00       400



## Create Submission

In [94]:
test_predictions = trainer.predict(test_dataset)

test_logits = test_predictions.predictions

test_probs = softmax(
    test_logits,
    axis=1
)

In [95]:
top3 = np.argsort(
    -test_probs,
    axis=1
)[:, :3]

In [96]:
### Convert IDs → Labels
top3_labels = []

for row in top3:

    prediction = " ".join(
        CFG.ID2LABEL[idx]
        for idx in row
    )

    top3_labels.append(prediction)

In [97]:
submission = sample.copy()

submission["Prediction"] = top3_labels

submission.head()

,ID,Prediction
0,1,A D C
1,2,B E D
2,3,B D E
3,4,E A D
4,5,C A E


In [98]:
submission["Prediction"].value_counts().head(20)

Prediction
B D E    91
C A E    75
A C E    43
D B A    37
C A D    30
A C D    27
E A D    23
D A B    21
B E D    19
E D A    17
E B D    15
C A B    12
D B E    12
D E B    11
A E C    10
E D B     9
D B C     6
E B A     6
D A E     5
E A C     5
Name: count, dtype: int64

In [99]:
submission.to_csv(
    "submission.csv",
    index=False
)

print("submission.csv created successfully.")

submission.csv created successfully.


## Save the models

In [100]:

trainer.save_model("./distilbert_baseline")

tokenizer.save_pretrained("./distilbert_baseline")

print("Model saved successfully.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully.
